# Gold Layer — Aggregations
Four business-level tables written as Parquet to HDFS:
1. `traffic_by_hour` — requests + bytes per day/hour, split by bot/human
2. `top_pages` — top 50 paths by human request count
3. `error_rates` — error request count and rate per day/hour
4. `bot_vs_human` — daily summary: bot requests, human requests, bot %

In [1]:
import os, sys
os.environ['SPARK_HOME'] = '/usr/local/spark-3.5.0-bin-hadoop3'
os.environ['HADOOP_USER_NAME'] = 'root'
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python')
sys.path.insert(0, '/usr/local/spark-3.5.0-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip')
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('04_gold')
    .master('local[4]')
    .config('spark.hadoop.fs.defaultFS', 'hdfs://hdfs-namenode:9000')
    .config('spark.sql.shuffle.partitions', '16')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)


Spark version: 3.5.0


In [2]:
silver = spark.read.parquet('hdfs://hdfs-namenode:9000/data/silver/access_logs')
print(f'Silver rows: {silver.count():,}')


Silver rows: 10,365,077


In [3]:
# ── Gold 1: Traffic by hour ──────────────────────────────────────────────────
traffic_by_hour = (
    silver
    .groupBy('log_date', 'hour', 'is_bot')
    .agg(
        F.count('*').alias('requests'),
        F.sum('bytes').alias('total_bytes'),
        F.avg('bytes').alias('avg_bytes'),
        F.countDistinct('ip').alias('unique_ips')
    )
    .orderBy('log_date', 'hour', 'is_bot')
)

traffic_by_hour.write.mode('overwrite').parquet(
    'hdfs://hdfs-namenode:9000/data/gold/traffic_by_hour'
)
print('traffic_by_hour written.')
spark.read.parquet('hdfs://hdfs-namenode:9000/data/gold/traffic_by_hour').show(10)


traffic_by_hour written.
+----------+----+------+--------+-----------+------------------+----------+
|  log_date|hour|is_bot|requests|total_bytes|         avg_bytes|unique_ips|
+----------+----+------+--------+-----------+------------------+----------+
|2019-01-22|   0| false|    4850|   83263817| 17167.79731958763|       264|
|2019-01-22|   0|  true|    3222|   87708848|27221.864680322782|       275|
|2019-01-22|   1| false|    7560|  112715403|14909.444841269842|       352|
|2019-01-22|   1|  true|    7355|  203020575|27603.069340584636|       391|
|2019-01-22|   2| false|    8403|  112804078|13424.262525288588|       536|
|2019-01-22|   2|  true|    7615|  223589181|29361.678397898882|       375|
|2019-01-22|   3| false|   19969|  236126560|11824.656217136562|      1252|
|2019-01-22|   3|  true|    8550|  218812487| 25592.10374269006|       410|
|2019-01-22|   4| false|   62486|  639960994|10241.670038088532|      2245|
|2019-01-22|   4|  true|   10133|  226793900|22381.713214250467

In [4]:
# ── Gold 2: Top 50 pages (human traffic only) ────────────────────────────────
# Strip query strings for cleaner grouping
top_pages = (
    silver
    .filter(~F.col('is_bot'))
    .withColumn('clean_path', F.regexp_replace('path', r'\?.*$', ''))
    .groupBy('clean_path')
    .agg(
        F.count('*').alias('requests'),
        F.countDistinct('ip').alias('unique_ips'),
        F.avg('bytes').alias('avg_bytes')
    )
    .orderBy(F.desc('requests'))
    .limit(50)
)

top_pages.write.mode('overwrite').parquet(
    'hdfs://hdfs-namenode:9000/data/gold/top_pages'
)
print('top_pages written.')
spark.read.parquet('hdfs://hdfs-namenode:9000/data/gold/top_pages').show(10, truncate=60)


top_pages written.
+------------------------------------------+--------+----------+------------------+
|                                clean_path|requests|unique_ips|         avg_bytes|
+------------------------------------------+--------+----------+------------------+
|                            /settings/logo|  351973|     96118| 4104.509027681101|
|                     /rapidGrails/jsonList|  196912|        59|3323.1315511497523|
|                  /site/alexaGooleAnalitic|  103685|     26328|  319.710527077205|
|         /static/css/font/wyekan/font.woff|  102770|     86773|  27571.1319645811|
|                              /favicon.ico|  102153|     67339|1.3252474229831723|
|/static/images/guarantees/goodShopping.png|   99023|     85664| 6299.668319481332|
|    /static/images/guarantees/warranty.png|   98352|     85575|  5621.07539246787|
|   /static/images/guarantees/bestPrice.png|   98095|     85516| 7126.010173811102|
|/static/images/guarantees/fastDelivery.png|   97692|    

In [5]:
# ── Gold 3: Error rates by day/hour ─────────────────────────────────────────
error_rates = (
    silver
    .groupBy('log_date', 'hour')
    .agg(
        F.count('*').alias('total_requests'),
        F.sum(F.col('is_error').cast('long')).alias('error_count'),
        F.round(
            100.0 * F.sum(F.col('is_error').cast('long')) / F.count('*'), 2
        ).alias('error_rate_pct')
    )
    .orderBy('log_date', 'hour')
)

error_rates.write.mode('overwrite').parquet(
    'hdfs://hdfs-namenode:9000/data/gold/error_rates'
)
print('error_rates written.')
spark.read.parquet('hdfs://hdfs-namenode:9000/data/gold/error_rates').show(10)


error_rates written.
+----------+----+--------------+-----------+--------------+
|  log_date|hour|total_requests|error_count|error_rate_pct|
+----------+----+--------------+-----------+--------------+
|2019-01-22|   0|          8072|        284|          3.52|
|2019-01-22|   1|         14915|        343|           2.3|
|2019-01-22|   2|         16018|        367|          2.29|
|2019-01-22|   3|         28519|        557|          1.95|
|2019-01-22|   4|         72619|        808|          1.11|
|2019-01-22|   5|        116544|       1339|          1.15|
|2019-01-22|   6|        148154|       2005|          1.35|
|2019-01-22|   7|        159590|       2192|          1.37|
|2019-01-22|   8|        160580|       2451|          1.53|
|2019-01-22|   9|        164233|       1874|          1.14|
+----------+----+--------------+-----------+--------------+
only showing top 10 rows



In [6]:
# ── Gold 4: Bot vs human daily summary ───────────────────────────────────────
bot_vs_human = (
    silver
    .groupBy('log_date')
    .agg(
        F.count('*').alias('total_requests'),
        F.sum(F.col('is_bot').cast('long')).alias('bot_requests'),
        F.sum((~F.col('is_bot')).cast('long')).alias('human_requests'),
        F.round(
            100.0 * F.sum(F.col('is_bot').cast('long')) / F.count('*'), 2
        ).alias('bot_pct'),
        F.countDistinct(
            F.when(~F.col('is_bot'), F.col('ip'))
        ).alias('unique_human_ips')
    )
    .orderBy('log_date')
)

bot_vs_human.write.mode('overwrite').parquet(
    'hdfs://hdfs-namenode:9000/data/gold/bot_vs_human'
)
print('bot_vs_human written.')
spark.read.parquet('hdfs://hdfs-namenode:9000/data/gold/bot_vs_human').show()


bot_vs_human written.
+----------+--------------+------------+--------------+-------+----------------+
|  log_date|total_requests|bot_requests|human_requests|bot_pct|unique_human_ips|
+----------+--------------+------------+--------------+-------+----------------+
|2019-01-22|       2160913|      220932|       1939981|  10.22|           55244|
|2019-01-23|       2328532|      266944|       2061588|  11.46|           56939|
|2019-01-24|       1857839|      233111|       1624728|  12.55|           57514|
|2019-01-25|       1843302|      243368|       1599934|   13.2|           60780|
|2019-01-26|       2174491|      166952|       2007539|   7.68|           46739|
+----------+--------------+------------+--------------+-------+----------------+



In [7]:
# Final HDFS inventory
print('=== HDFS Gold layer ===')
import subprocess
# Show all gold tables via spark
for tbl in ['traffic_by_hour','top_pages','error_rates','bot_vs_human']:
    df = spark.read.parquet(f'hdfs://hdfs-namenode:9000/data/gold/{tbl}')
    print(f'{tbl}: {df.count():,} rows')

spark.stop()
print('Gold layer done.')


=== HDFS Gold layer ===
traffic_by_hour: 228 rows
top_pages: 50 rows
error_rates: 114 rows
bot_vs_human: 5 rows
Gold layer done.
